In [2]:
%%sql

-- CTE Example

-- WITH                              -- opens the CTE block
--    cte_name AS (                 -- name your temporary result set
--        SELECT ...                -- the query that defines it
--        FROM ...
--    ),                            -- comma separates multiple CTEs
--    second_cte AS (               -- you can chain as many as you need
--        SELECT ...
--        FROM cte_name             -- CTEs can reference earlier CTEs
--    )
-- SELECT *                          -- main query references CTE by name
-- FROM second_cte

UsageError: Cell magic `%%sql` not found.


In [3]:
import pandas as pd
import numpy as np
%load_ext sql
%sql duckdb:///:memory:

np.random.seed(22)
suppliers = ["Acme", "GlobalCo", "FastParts", "PrimeMg", "EastCoast"]
categories = ["Electronics", "Hardware", "Consumables", "Apparel"]
warehouses = ["East", "West", "Central"]

rows = []
for i in range(1, 201):
    rows.append({
        "po_id":          f"PO-{str(i).zfill(4)}",
        "supplier":       np.random.choice(suppliers),
        "category":       np.random.choice(categories),
        "warehouse":      np.random.choice(warehouses),
        "order_date":     pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.randint(0, 365))),
        "amount":         round(np.random.uniform(500, 50000), 2),
        "units":          np.random.randint(10, 500),
        "lead_time_days": np.random.randint(3, 45),
        "on_time":        np.random.choice([True, False], p=[0.8, 0.2])
    })

po = pd.DataFrame(rows)
%sql --persist po
print(f"po: {po.shape}")

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

Success! Persisted po to the database.

po: (200, 9)


In [4]:
po.head(5)

,po_id,supplier,category,warehouse,order_date,amount,units,lead_time_days,on_time
0,PO-0001,EastCoast,Electronics,East,2024-12-22,32055.22,94,11,True
1,PO-0002,FastParts,Consumables,East,2024-02-15,23470.66,103,42,True
2,PO-0003,Acme,Electronics,West,2024-01-28,22382.64,39,21,True
3,PO-0004,GlobalCo,Apparel,Central,2024-01-08,47315.60,33,26,True
4,PO-0005,EastCoast,Apparel,East,2024-12-25,37347.63,123,4,False


In [ ]:
%%sql

-- S5.1 Single CTE
-- Business question: which suppliers spend above the network average?

-- example below is incorrect

WITH supplier_spend AS (
    SELECT
        supplier,
        ROUND(SUM(amount), 2) AS total_spend
    FROM
        po
    GROUP BY
        supplier
)
SELECT
    supplier,
    total_spend,
    ROUND(AVG(total_spend) OVER (), 2) AS network_avg,                  -- reference total_spend from CTE calculates on filtered rows
    ROUND(total_spend - AVG(total_spend) OVER (), 2) AS difference      -- reference total_spend from CTE
FROM
    supplier_spend                                                      -- reference the CTE
WHERE
    total_spend > (SELECT AVG(total_spend) FROM supplier_spend)         -- filters rows, note: WHERE runs before window functions
ORDER BY
    total_spend DESC



Running query in 'duckdb:///:memory:'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
(_duckdb.ParserException) Parser Error: syntax error at or near "WITH"

LINE 24: WITH supplier_spend AS (
         ^
[SQL: WITH supplier_spend AS (
    SELECT
        supplier,
        ROUND(SUM(amount), 2) AS total_spend
    FROM
        po
    GROUP BY
        supplier
)
SELECT
    supplier,
    total_spend,
    ROUND(AVG(total_spend) OVER (), 2) AS network_avg,
    ROUND(total_spend - AVG(total_spend) OVER (), 2) AS difference
FROM
    supplier_spend
WHERE
    total_spend > (SELECT AVG(total_spend) FROM supplier_spend)
ORDER BY
    total_spend DESC



WITH supplier_spend AS (
    SELECT supplier, ROUND(SUM(amount), 2) AS total_spend
    FROM po GROUP BY supplier
),
with_avg AS (
    SELECT supplier, total_spend,
           ROUND(AVG(total_spend) OVER(), 2) AS network_avg
